In [11]:
# !pip install curl_cffi
sleep = 10

In [12]:
from curl_cffi import requests
import json
import time
time.sleep(sleep)
# 1. Загрузка robots.txt
url = 'https://www.ozon.ru/robots.txt'
with open('headers.json', 'r', encoding='utf-8') as file:
    headers = json.load(file)

try:
    response = requests.get(url, headers=headers, impersonate='chrome120', timeout=10)
    print(response.status_code)
    robots_content = response.text
except Exception as e:
    print(f"Ошибка: {e}")

200


In [13]:
from urllib.robotparser import RobotFileParser


time.sleep(sleep)
# 2. Использование RobotFileParser для проверки конкретных URL
rp = RobotFileParser()
rp.parse(robots_content.splitlines())

# Задаём имя нашего бота
BOT_NAME = '*'  # Мы не гугл и не яндекс, так что задаем общее название 

# Список важных разделов для проверки
test_urls = [
    'https://www.ozon.ru/',
    'https://www.ozon.ru/category/',
    'https://www.ozon.ru/category/elektronika-15500/',
    'https://www.ozon.ru/category/smartfony-15502/',
    'https://www.ozon.ru/product/oppo-smartfon-a5-pro-rostest-eac-8-256-gb-olivkovyy-1936982238/?at=WPtNYjAG2ILDQKZWcjVZ39nHRPOpzKUkkOk0jSP72Dlk',
    # 'https://www.ozon.ru/',
    # 'https://www.ozon.ru/',
    # 'https://www.ozon.ru/',
]

print("--- Проверка доступности разделов для бота '%s' ---" % BOT_NAME)
for test_url in test_urls:
    can_fetch = rp.can_fetch(BOT_NAME, test_url)
    status = "РАЗРЕШЕНО" if can_fetch else "ЗАПРЕЩЕНО"
    print(f"{status}: {test_url}")

# Задержка
delay = rp.crawl_delay(BOT_NAME)
print(f"\nРекомендуемая задержка (Crawl-delay) для бота '{BOT_NAME}': {delay} сек.")

# Карты сайта
sitemaps = rp.site_maps()
print("Карты сайта (Sitemap):", sitemaps)
 

--- Проверка доступности разделов для бота '*' ---
РАЗРЕШЕНО: https://www.ozon.ru/
РАЗРЕШЕНО: https://www.ozon.ru/category/
РАЗРЕШЕНО: https://www.ozon.ru/category/elektronika-15500/
РАЗРЕШЕНО: https://www.ozon.ru/category/smartfony-15502/
РАЗРЕШЕНО: https://www.ozon.ru/product/oppo-smartfon-a5-pro-rostest-eac-8-256-gb-olivkovyy-1936982238/?at=WPtNYjAG2ILDQKZWcjVZ39nHRPOpzKUkkOk0jSP72Dlk

Рекомендуемая задержка (Crawl-delay) для бота '*': None сек.
Карты сайта (Sitemap): None


#### Итого:  
Для нашего бота доступны как страницы категорий, так и страницы товаров. Задержка не указана, поэтому для запросов будет использоваться задержка в 5 секунд

In [14]:
from bs4 import BeautifulSoup
import json
import re
import time

def fetch_categories_ozon(url, headers):
    """
    Пытается извлечь структуру категорий с главной страницы каталога Ozon.
    Возвращает словарь с названиями и URL категорий или пустой словарь в случае ошибки.
    """
    time.sleep(sleep)

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # Используем BeautifulSoup для парсинга HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        categories = {}
        
        # Ищем все ссылки, которые ведут на страницы категорий
        for link in soup.find_all('a', href=True):
            href = link['href']
            # Фильтруем ссылки, которые похожи на категории
            if '/category/' in href and '/brand/' not in href and '?' not in href and not href.startswith('http'):
                full_url = 'https://www.ozon.ru' + href
                # Берём текст ссылки как название
                name = link.get_text(strip=True)
                if name and len(name) > 1:  # Отсекаем пустые или служебные
                    categories[name] = full_url
        
        return categories
    
    except requests.exceptions.RequestException as e:
        print(f"Ошибка загрузки страницы: {e}")
        return {}
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")
        return {}



In [15]:

url = 'https://www.ozon.ru/category'
categories = fetch_categories_ozon(url, headers)

if categories:
    for i, (name, url) in enumerate(categories.items()):
        print(f"  {name}: {url}")

  Одежда: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/
  Электроника: https://www.ozon.ru/category/elektronika-15500/
  Дом и сад: https://www.ozon.ru/category/dom-i-sad-14500/
  Детские товары: https://www.ozon.ru/category/detskie-tovary-7000/
  Одежда, обувь и аксессуары: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/
  Аптека: https://www.ozon.ru/category/apteka-6000/
  Бытовая техника: https://www.ozon.ru/category/bytovaya-tehnika-10500/
  Книги: https://www.ozon.ru/category/knigi-16500/
  Строительство и ремонт: https://www.ozon.ru/category/stroitelstvo-i-remont-9700/
  Красота и здоровье: https://www.ozon.ru/category/krasota-i-zdorove-6500/
  Спортивные товары: https://www.ozon.ru/category/sport-i-otdyh-11000/
  Товары для животных: https://www.ozon.ru/category/tovary-dlya-zhivotnyh-12300/
  Продукты питания: https://www.ozon.ru/category/produkty-pitaniya-9200/
  Автотовары: https://www.ozon.ru/category/avtotovary-8500/
  Товары для взрослыхТовары дл

In [16]:
def keep_first_keys_by_value(d):
    """
    Возвращает новый словарь, содержащий только первые ключи для каждого уникального значения.
    Сохраняет порядок первого появления ключей.
    """
    seen_values = set()
    result = {}
    for key, value in d.items():
        if value not in seen_values:
            seen_values.add(value)
            result[key] = value
    return result

In [17]:
subcategories = {}
for name, url in categories.copy().items():
    categories_update = fetch_categories_ozon(url, headers)
    if len(categories_update) > 0:
        subcategories.update(categories_update)
    else:
        subcategories.update({name : url})

subcategories = keep_first_keys_by_value(subcategories)

if subcategories:
    for i, (name, url) in enumerate(subcategories.items()):
        print(f"  {name}: {url}")

  Одежда: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/
  Электроника: https://www.ozon.ru/category/elektronika-15500/
  Дом и сад: https://www.ozon.ru/category/dom-i-sad-14500/
  Женщинам: https://www.ozon.ru/category/zhenskaya-odezhda-7501/
  Мужчинам: https://www.ozon.ru/category/muzhskaya-odezhda-7542/
  Детям: https://www.ozon.ru/category/detskaya-odezhda-7580/
  Уход за одеждой: https://www.ozon.ru/category/sredstva-dlya-uhoda-za-odezhdoy-7757/
  Униформа: https://www.ozon.ru/category/spetsodezhda-i-sredstva-individualnoy-zashchity-10189/
  Верхняя одежда: https://www.ozon.ru/category/verhnyaya-odezhda-muzhskaya-7543/
  Толстовки: https://www.ozon.ru/category/tolstovki-i-olimpiyki-muzhskie-7555/
  Деним: https://www.ozon.ru/category/dzhinsy-muzhskie-7556/
  Нижнее бельё: https://www.ozon.ru/category/nizhnee-bele-muzhskoe-7578/
  Футболки: https://www.ozon.ru/category/futbolki-i-polo-muzhskie-7558/
  Бытовая техника: https://www.ozon.ru/category/bytovaya-tehnika-105

In [18]:
# import requests
from lxml import html

def clean_price(val):
    """Удаляет тонкие пробелы и знак рубля, преобразует в число (int/float)."""
    if val is None:
        return None
    cleaned = val.replace('\u2009', '').replace('₽', '').strip()
    try:
        # Если есть точка — float, иначе int
        return float(cleaned) if '.' in cleaned else int(cleaned)
    except ValueError:
        # Если не удалось преобразовать, возвращаем очищенную строку
        return cleaned

def extract_number(val):
    """Извлекает первое число из строки (например, для количества отзывов)."""
    if val is None:
        return None
    match = re.search(r'\d+', val)
    return int(match.group()) if match else None

def clean_str(val):
    """Просто удаляет тонкие пробелы и знак рубля, оставляя строку."""
    if val is None:
        return None
    return val.replace('\u2009', '').replace('₽', '').strip()


def parse_products_page_lxml(url, headers, product_xpath, field_xpaths, post_processors=None):
    time.sleep(sleep)
    try:
        response = requests.get(url, headers=headers, impersonate='chrome120', timeout=10)
        response.raise_for_status()
    
        dom = html.fromstring(response.text)
        product_nodes = dom.xpath(product_xpath)
                                    
        results = []
        for node in product_nodes:
            product_data = {}
            for field_name, xpath_expr in field_xpaths.items():
                # Извлекаем данные из текущего блока товара
                extracted = node.xpath(xpath_expr)
                
                # Обработка результата: обычно берём первый непустой текст
                if extracted:
                    # Если результат — список строк или элементов, собираем текст
                    if all(isinstance(e, str) for e in extracted):
                        value = ' '.join(extracted).strip()
                    else:
                        # Для элементов lxml получаем текстовое содержимое
                        value = ' '.join([e.text_content().strip() for e in extracted if hasattr(e, 'text_content')])
                    product_data[field_name] = value
                else:
                    product_data[field_name] = None

                if post_processors and field_name in post_processors:
                    try:
                        value = post_processors[field_name](value)
                    except Exception as e:
                        print(f"Ошибка обработки поля {field_name} со значением {value}: {e}")
                        value = None
                product_data[field_name] = value
            
            results.append(product_data)
        
        return results
    except requests.exceptions.RequestException as e:
        print(f"Ошибка загрузки страницы: {e}")
        return {}
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")
        return {}


url = "https://www.ozon.ru/category/nabory-dlya-issledovaniy-i-opytov-13886/?page=1"

product_xpath="//div[contains(@class, 'qi0_24')]/div[contains(@class, 'tile-root')]"
field_xpaths = {
    "image_href": ".//a[contains(@class, 'q4b1_4_0-a tile-clickable-element ki1_24')]/@href",
    "name": ".//span[contains(@class, 'tsBody500Medium')]/text()",
    "cost": ".//span[contains(@class, 'c35_3_13-a1 tsHeadline500Medium')]/text()",
    "discount": ".//span[contains(@class, 'tsBodyControl400Small c35_3_13-a6 c35_3_13-b4')]/text()",
    "rating": ".//div[contains(@class, 'i4k_24 ik5_24 p6b3_2_0-a p6b3_2_0-a0 p6b3_2_0-a1 tsBodyMBold')]/span[contains(@class, 'p6b3_2_0-a4')][1]/span/text()",
    "reviews": ".//div[contains(@class, 'i4k_24 ik5_24 p6b3_2_0-a p6b3_2_0-a0 p6b3_2_0-a1 tsBodyMBold')]/span[contains(@class, 'p6b3_2_0-a4')][2]/span/text()",
}
post_processors = {
    "image_href" : clean_str,
    "name"       : clean_str,
    "cost"       : clean_price,
    "discount"   : extract_number,
    "rating"     : clean_price,
    "reviews"    : extract_number,
}

parse_products_page_lxml(url, headers, product_xpath, field_xpaths, post_processors)

[{'image_href': '/product/detskiy-mikroskop-m6-3026337010/?at=16tLmw9LkUmKq5B7spPJNVPU380MKgtRPglXYt51MwQ4',
  'name': 'Детский микроскоп M6',
  'cost': 1650,
  'discount': 79,
  'rating': 4.8,
  'reviews': 5},
 {'image_href': '/product/mikroskop-shkolnyy-detskiy-s-naborom-dlya-opytov-2917620223/?at=28t0Go90RuqPrg7f7MOxlXSoENrBoukZKW2Oi386kZ',
  'name': 'Микроскоп школьный детский с набором для опытов',
  'cost': 1529,
  'discount': 68,
  'rating': 4.8,
  'reviews': 5},
 {'image_href': '/product/opyty-i-eksperimenty-dlya-detey-14-v-1-nabor-opytov-dlya-malchikov-i-devochek-podarok-dlya-malchika-2419096711/?at=QktJq4yJ3c59VBkwCwOv2EvsAzP6KniP2rKnVsQlK0mr',
  'name': 'Опыты и эксперименты для детей "14 в 1", набор опытов для мальчиков и девочек, подарок для мальчика и девочки',
  'cost': 1233,
  'discount': 74,
  'rating': 4.9,
  'reviews': 26},
 {'image_href': '/product/opyty-i-eksperimenty-dlya-detey-gologramma-podarok-dlya-malchika-i-devochki-168988996/?at=DqtDmrkDnu5QwDnWHKDP5K1IKJ8B4

In [19]:
import time
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode

def collect_all_products(subcategories, product_xpath, field_xpaths, post_processors=None, max_pages=10, delay=5):
    """
    Собирает товары из всех подкатегорий с пагинацией.

    Параметры:
        subcategories (dict): Словарь вида {'Название категории': 'https://ozon.ru/category/...'}
        base_xpaths (dict): Словарь с XPath. Должен содержать:
            - 'product': XPath для поиска блоков товаров.
            - 'fields': словарь {имя_поля: относительный_XPath} для извлечения данных из блока.
        post_processors (dict, optional): Словарь функций для обработки значений полей.
        max_pages (int): Максимальное количество страниц для сканирования (по умолчанию 10).
        delay (int): Задержка между запросами в секундах (соблюдение crawl-delay).

    Возвращает:
        list: Список словарей с данными товаров. Каждый словарь содержит поля товара,
              а также метаданные: 'category_name', 'page_url', 'page_number'.
    """
    all_products = []  # итоговая структура данных


    for cat_name, cat_url in subcategories.items():
        print(f"\n--- Обработка категории: {cat_name} ---")

        for page in range(1, max_pages + 1):
            # Формируем URL с параметром page
            parsed = urlparse(cat_url)
            query = parse_qs(parsed.query)
            query['page'] = [str(page)]  # заменяем или добавляем page
            new_query = urlencode(query, doseq=True)
            new_parsed = parsed._replace(query=new_query)
            page_url = urlunparse(new_parsed)

            print(f"  Загрузка страницы {page}: {page_url}")

            # Вызываем парсер для одной страницы
            try:
                products = parse_products_page_lxml(page_url, headers, product_xpath, field_xpaths, post_processors)
            except Exception as e:
                print(f"  ❌ Ошибка при парсинге страницы: {e}")
                break  # если страница не грузится, возможно, пагинация прервалась

            # Если товаров нет, прекращаем пагинацию для этой категории
            if not products:
                print(f"  Страница {page} не содержит товаров. Завершаем категорию.")
                break

            # Добавляем метаданные и сохраняем
            for prod in products:
                prod['category_name'] = cat_name
                prod['page_url'] = page_url
                prod['page_number'] = page
                all_products.append(prod)

            print(f"  ✅ Добавлено товаров: {len(products)}")

            # Задержка перед следующим запросом
            if page < max_pages:  # после последней страницы задержка не нужна
                time.sleep(delay)

    print(f"\nВсего собрано товаров: {len(all_products)}")
    return all_products

In [20]:
all_data = collect_all_products(
    subcategories,
    product_xpath, 
    field_xpaths,
    post_processors,
    max_pages=6,      # максимум 5 страниц для теста
    delay=5           # пауза 5 секунд между запросами
)

# Теперь all_data можно сохранить в БД, CSV, JSON и т.д.
# Например, запись в JSON:
import json
with open('ozon_products_new.json', 'w', encoding='utf-8') as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)


--- Обработка категории: Одежда ---
  Загрузка страницы 1: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/?page=1
Ошибка обработки поля discount со значением 2621: expected string or bytes-like object, got 'int'
  ✅ Добавлено товаров: 8
  Загрузка страницы 2: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/?page=2
Ошибка обработки поля discount со значением 222: expected string or bytes-like object, got 'int'
Ошибка обработки поля discount со значением 772: expected string or bytes-like object, got 'int'
Ошибка обработки поля discount со значением 423: expected string or bytes-like object, got 'int'
  ✅ Добавлено товаров: 8
  Загрузка страницы 3: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/?page=3
Ошибка обработки поля discount со значением 644: expected string or bytes-like object, got 'int'
  ✅ Добавлено товаров: 8
  Загрузка страницы 4: https://www.ozon.ru/category/odezhda-obuv-i-aksessuary-7500/?page=4
Ошибка обработки поля discount со знач